In [3]:
import hail as hl
#import argparse
#import pandas
import os

from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [5]:
import hail as hl
from gnomad.utils.vep import process_consequences
import os

In [6]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('/logs/hail/hail_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe001.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.74-0c3a74d12093
LOGGING: writing to /logs/hail/hail_export.log


In [43]:
# get non singletons (note: individuals are on 12788)
mt1 = qc.get_table('data/phased/ukb_wes_200k_phased_chr22.1of1.vcf.gz', 'vcf')
#mt1 = qc.get_table('data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1.vcf.gz', 'vcf')
#mt1 = qc.recalc_info(mt1)
#mt1 = qc.translate_sample_ids(mt1, from_app='12788', to_app='11867') # translate to lindgren app ID
#mt1 = qc.filter_to_european(mt1)
mt1 = qc.filter_min_missing(mt1, 0.05)
mt1 = qc.filter_max_maf(mt1, float(0.02))
mt1 = analysis.annotate_vep(mt1, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
mt1 = analysis.filter_vep(mt1, 'consequence_category',['damaging_missense','ptv'])
#mt1 = mt1.filter_rows(hl.agg.any(mt1.GT.is_non_ref()))

Annotating with VEP file: data/vep/full/ukb_wes_200k_full_vep_chr22.vcf


In [44]:
mt2 = qc.get_table('data/unphased/unfiltered/ukb_wes_200k_filtered_chr22.mt','mt')
#mt2 = qc.get_table('data/unphased/tmp/ukb_wes_200k_filtered_2samples_chr22.vcf.bgz', 'vcf')
mt2 = qc.filter_to_european(mt2)
mt2 = qc.filter_max_mac(mt2, 1) # get singletons
mt2 = qc.filter_min_missing(mt2, 0.05)
mt2 = analysis.annotate_vep(mt2, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
mt2 = analysis.filter_vep(mt2, 'consequence_category',['damaging_missense','ptv'])


2021-09-16 23:02:25 Hail: INFO: Reading table without type imputation
  Loading field 'eid' as type str (user-supplied)
  Loading field 'genetically_european' as type int32 (user-supplied)


[get_genetically_european]: Not all samples IDs mapped perfectly (522/199795 IDs are undefined)
[get_genetically_european]:187330/199795 IDs were included as genetically european.
Annotating with VEP file: data/vep/full/ukb_wes_200k_full_vep_chr22.vcf


In [45]:
# Count burden per gene per individual
mt1_cat = analysis.gene_burden_category_annotations_per_sample(mt1)
mt2_cat = analysis.gene_burden_category_annotations_per_sample(mt2)

# combine singleton table and full table
res = mt1_cat.annotate_entries(singletons = mt2_cat[(mt1_cat.Gene, mt1_cat.consequence_category), mt1_cat.s].n)
res = res.annotate_entries(singletons = hl.if_else(hl.is_missing(res.singletons),0,res.singletons))
res = res.annotate_entries(total = res.n + res.singletons)
res = res.entries()
res = res.filter(res.total > 0)

In [46]:
n = res.singletons.collect()

2021-09-16 23:04:15 Hail: INFO: Coerced almost-sorted dataset
2021-09-16 23:04:17 Hail: INFO: Coerced sorted dataset
2021-09-16 23:21:04 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-16 23:21:07 Hail: INFO: Coerced sorted dataset
2021-09-17 00:00:13 Hail: INFO: Ordering unsorted dataset with network shuffle


In [47]:
n

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [6]:
# why are there no singletons left in chr 22 after filtering?

# How many singletons / individuals at missingness 5%
mmt2 = mt2.filter_rows(hl.agg.any(mt2.GT.is_non_ref()))
#mmt1 = mt1.filter_rows(hl.agg.any(mt1.GT.is_non_ref()))

# 

In [ ]:
mmt2.count()

2021-09-16 13:58:13 Hail: INFO: Coerced sorted dataset


In [ ]:
# Count burden per gene per individual
mt1_cat = analysis.gene_burden_category_annotations_per_sample(mt1)
mt2_cat = analysis.gene_burden_category_annotations_per_sample(mt2)

# combine singleton table and full table
res = mt1_cat.annotate_entries(singletons = mt2_cat[(mt1_cat.Gene, mt1_cat.consequence_category), mt1_cat.s].n)
res = res.annotate_entries(singletons = hl.if_else(hl.is_missing(res.singletons),0,res.singletons))
res = res.annotate_entries(total = res.n + hl.if_else(hl.is_missing(res.singletons),0,res.singletons))
res = res.entries()
res = res.filter(res.singletons > 0)

In [26]:
mt = analysis.get_prob_ko_matrix(mt1, mt2, field_drop = None)

2021-09-11 12:51:20 Hail: INFO: Coerced almost-sorted dataset
2021-09-11 12:51:26 Hail: INFO: Coerced sorted dataset
2021-09-11 12:51:26 Hail: INFO: Coerced almost-sorted dataset
2021-09-11 12:51:28 Hail: INFO: Coerced sorted dataset
2021-09-11 12:51:43 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-09-11 12:51:44 Hail: INFO: Coerced sorted dataset
2021-09-11 12:51:45 Hail: INFO: Coerced sorted dataset
2021-09-11 12:51:58 Hail: INFO: Ordering unsorted dataset with network shuffle


+-------------------+----------------------+---------------+----------------------+---------------+
| Gene              | '1705888'.singletons | '1705888'.pKO | '3544034'.singletons | '3544034'.pKO |
+-------------------+----------------------+---------------+----------------------+---------------+
| str               |                int64 |       float64 |                int64 |       float64 |
+-------------------+----------------------+---------------+----------------------+---------------+
| ""                |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000008735" |                    1 |      0.00e+00 |                    2 |      5.00e-01 |
| "ENSG00000015475" |                    1 |      0.00e+00 |                    1 |      0.00e+00 |
| "ENSG00000025770" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000040608" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000054611" |                    0 |      0.00e+00 |                    1 |      0.00e+00 |
| "ENSG00000056487" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000063515" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000069998" |                    0 |      0.00e+00 |                    2 |      5.00e-01 |
| "ENSG00000070010" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000070371" |                    0 |      0.00e+00 |                    1 |      0.00e+00 |
| "ENSG00000070413" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000073146" |                    3 |      7.50e-01 |                   13 |      1.00e+00 |
| "ENSG00000073150" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000073169" |                    2 |      5.00e-01 |                    1 |      0.00e+00 |
| "ENSG00000075218" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000075234" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000075240" |                    3 |      7.50e-01 |                    3 |      7.50e-01 |
| "ENSG00000075275" |                    1 |      0.00e+00 |                    9 |      9.96e-01 |
| "ENSG00000077935" |                    2 |      5.00e-01 |                    2 |      5.00e-01 |
| "ENSG00000077942" |                    2 |      5.00e-01 |                    2 |      5.00e-01 |
| "ENSG00000079974" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000093000" |                    2 |      5.00e-01 |                    0 |      0.00e+00 |
| "ENSG00000093009" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000093010" |                    1 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000093072" |                    2 |      5.00e-01 |                    1 |      0.00e+00 |
| "ENSG00000099889" |                    0 |      0.00e+00 |                    4 |      8.75e-01 |
| "ENSG00000099901" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000099904" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000099910" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000099917" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000099937" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000099940" |                    0 |      0.00e+00 |                    2 |      5.00e-01 |
| "ENSG00000099942" |                    0 |      0.00e+00 |                    0 |      0.00e+00 |
| "ENSG00000099949" |                    0 |      0.00e+00 |                    1 |      0.00e+00 |


In [38]:
mt = qc.get_table('data/mt/ukb_wes_vep_200_chr22.mt','mt')

In [42]:
mt.vep.decribe()

AttributeError: StructExpression instance has no field, method, or property 'decribe'
    Did you mean:
        StructExpression inherited method: 'describe'

In [27]:
# annotate with VEP
#hail_init.hail_bmrc_init('/logs/hail/hail_vep_export.log', 'GRCh38')
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
dataset = qc.get_table('data/unphased/unfiltered/ukb_wes_200k_filtered_chr2.mt', 'mt')
    
# clean up snpID and rsID
dataset = qc.annotate_snpid(dataset)
dataset = qc.annotate_rsid(dataset)
dataset = qc.default_to_snpid_when_missing_rsid(dataset)

# Translate to lindgren IDs
#if True:
#    dataset = qc.translate_sample_ids(dataset, 12788, 11867)    
    
# recalc info
dataset = qc.recalc_info(dataset)

# get VEP
result = hl.vep(dataset, "utils/configs/vep_env.json") 
result = process_consequences(result)
qc.export_table(result, out_prefix = 'testtabledeleteme', out_type = 'mt')


2021-09-16 20:44:25 Hail: WARN: expected input file 'file:/well/lindgren/flassen/ressources/dbsnp/GRCh38/GCF_000001405.39.gz' to end in .vcf[.bgz, .gz]



Writing to MT: testtabledeleteme.mt



KeyboardInterrupt: 

In [26]:
#result.rsid.show()

KeyboardInterrupt: 

ModuleNotFoundError: No module named 'ukb_common'